# Plots -- "Heterogeneity Gap and Failure-Mode Analysis"

Generates every figure for the first results section. **Training/eval curves
only, no bar charts.** Lead curves are the **optimality gap to the BS-Optimized
ceiling** (`% above optimal`, lower = better) via `compute_gap.gap_curve`, with
the pre-registered **normalized-score** versions kept as a companion section at
the end. Failure modes are per-policy rollout panels from `eval_rollout_plots`.

**Workflow:** run top-to-bottom. The curve figures (P1, P2) run on existing
data. The failure-mode rollouts (P3) need the re-tuned MAPPO-PS `--visualize`
rollouts -- paste their paths into `ROLLOUTS` in the SETUP cell and re-run;
until then those cells skip cleanly. Figures are saved to
`experiment_outputs/figures/`.

In [ ]:
# === SETUP: paths + the one thing you paste ==================================
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys, warnings
warnings.filterwarnings("ignore")

# Notebook lives in src/<analysis|visualization>/, so repo root is two parents up.
REPO_ROOT = Path.cwd().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Outputs are written here, never into thesis/ (copy them over yourself).
OUT_DIR = REPO_ROOT / "experiment_outputs" / "analysis_outputs"
FIG_DIR = REPO_ROOT / "experiment_outputs" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ---- Environments -----------------------------------------------------------
ENVS   = ["symmetric", "sku_hetero", "demand_hetero", "cost_hetero", "combined_hetero"]
HETERO = ["sku_hetero", "demand_hetero", "cost_hetero", "combined_hetero"]
ENV_TOK   = {"symmetric": "Symmetric", "sku_hetero": "SKUHetero",
             "demand_hetero": "DemandHetero", "cost_hetero": "CostHetero",
             "combined_hetero": "CombinedHetero"}
ENV_LABEL = {"symmetric": "Symmetric", "sku_hetero": "SKU", "demand_hetero": "Demand",
             "cost_hetero": "Cost", "combined_hetero": "Combined"}

# ---- Algorithms in the story (noPS axis cut) --------------------------------
# CPPO is the centralized RL ceiling. Re-add "MAPPO-noPS"/"IPPO-noPS" here only
# if you later run them on the hetero envs.
ALGOS = ["MAPPO-PS", "IPPO-PS", "CPPO"]
BASELINE_KINDS = ["Random", "ConstantOrder", "BSNewsvendor", "BSAdaptive",
                  "BSOptimized", "BSIndependent"]

# ====== PASTE HERE ===========================================================
# Re-tuned MAPPO-PS rollouts produced with run_evaluation.sh --visualize.
# Each value is a directory that DIRECTLY contains episodes_data.npz
# (E1.5 locked MAPPO-PS per env; symmetric is the E0.3 reference column).
# Leave None until ready -> dependent cells skip cleanly.
ROLLOUTS = {
    "symmetric":       None,   # E0.3 locked MAPPO-PS  (reference column)
    "sku_hetero":      None,   # E1.5 SKUHetero        (MAIN: per-SKU collapse)
    "demand_hetero":   None,   # E1.5 DemandHetero     (MAIN: per-warehouse scale)
    "cost_hetero":     None,   # E1.5 CostHetero       (appendix)
    "combined_hetero": None,   # E1.5 CombinedHetero   (appendix)
}

# ---- Directory registry (pre-filled to the on-disk E0/E1 layout) ------------
TUNING = REPO_ROOT / "experiment_outputs" / "Tuning"

def _locked(seed_eval_dir):
    """load_run config dir: locked '01_*' trial (tune mode) else the dir itself
    (single-config / transfer mode)."""
    d = Path(seed_eval_dir)
    if not d.is_dir():
        return None
    ranked = sorted(d.glob("01_*"))
    return ranked[0] if ranked else d

_E0 = TUNING / "E0"
SYMM_RL = {
    "MAPPO-PS":   _E0 / "MAPPO_PS_Tune_3WH_5SKUS_Symmetric_Optuna_FIFO_E0"   / "seed_evaluation",
    "IPPO-PS":    _E0 / "IPPO_PS_Tune_3WH_5SKUS_Symmetric_Optuna_FIFO_E0"    / "seed_evaluation",
    "CPPO":       _E0 / "CPPO_Tune_3WH_5SKUS_Symmetric_Optuna_FIFO_E0"       / "seed_evaluation",
    "MAPPO-noPS": _E0 / "MAPPO_NoPS_Tune_3WH_5SKUS_Symmetric_Optuna_FIFO_E0" / "seed_evaluation",
    "IPPO-noPS":  _E0 / "IPPO_NoPS_Tune_3WH_5SKUS_Symmetric_Optuna_FIFO_E0"  / "seed_evaluation",
}
SYMM_BASE = _E0 / "BASELINES_Tune_3WH_5SKUS_Symmetric_E0" / "seed_evaluation"

_E11 = TUNING / "E1" / "E1.1_Hetero_Transfer"
_TG = {"MAPPO-PS": "MAPPO_PS", "IPPO-PS": "IPPO_PS", "CPPO": "CPPO"}
def transfer_dir(algo, env):
    a = _TG.get(algo)
    if a is None:
        return None
    return (_E11 / f"{a}_HeteroTransfer_E1.1"
                 / f"{a}_SeedEval_3WH_5SKUS_{ENV_TOK[env]}_E1.1" / "seed_evaluation")

def retune_dir(algo, env):
    tok = ENV_TOK[env]
    if algo == "CPPO":
        return TUNING/"E1"/"E1.3_Hetero_CPPO_Retune"/f"CPPO_Tune_3WH_5SKUS_{tok}_Optuna_FIFO_E1.3"/"seed_evaluation"
    if algo == "IPPO-PS":
        return TUNING/"E1"/"E1.4_Hetero_IPPO_Retune"/f"IPPO_PS_Tune_3WH_5SKUS_{tok}_Optuna_FIFO_E1.4"/"seed_evaluation"
    if algo == "MAPPO-PS":
        return TUNING/"E1"/"E1.5_Hetero_MAPPO_Retune"/f"MAPPO_PS_Tune_3WH_5SKUS_{tok}_Optuna_FIFO_E1.5"/"seed_evaluation"
    return None

_E12 = TUNING/"E1"/"E1.2_Hetero_Baselines"/"BASELINES_3WH5SKUS_Hetero_E1.2"
def baseline_dir(env, kind):
    if env == "symmetric":
        return SYMM_BASE / kind
    return _E12 / f"BASELINES_3WH5SKUS_{ENV_TOK[env]}" / "seed_evaluation" / kind

print("REPO_ROOT :", REPO_ROOT)
print("tables ->", OUT_DIR)
print("figures ->", FIG_DIR)


In [ ]:
# === LOAD: read every run once (graceful about missing dirs) =================
from src.visualization.utils.plot_data_loader import load_run, load_episodes

def _resolve(p):
    p = Path(p)
    return _locked(p) if p.name == "seed_evaluation" else p

def _try_load(p, what):
    if p is None:
        return None
    rp = _resolve(p)
    if rp is None or not rp.is_dir():
        return None
    try:
        run = load_run(rp)
    except Exception as e:
        print(f"  ! load failed [{what}]: {e}")
        return None
    return run if run["seed_data"] else None

RL_SYMM     = {a: _try_load(SYMM_RL.get(a), f"symm/{a}") for a in ALGOS}
RL_TRANSFER = {e: {a: _try_load(transfer_dir(a, e), f"transfer/{a}/{e}") for a in ALGOS} for e in HETERO}
RL_RETUNE   = {e: {a: _try_load(retune_dir(a, e),   f"retune/{a}/{e}")   for a in ALGOS} for e in HETERO}
BASE        = {e: {k: _try_load(baseline_dir(e, k), f"base/{k}/{e}")     for k in BASELINE_KINDS} for e in ENVS}

def _n(run):
    return len(run["seed_data"]) if run else 0

print("== coverage (n seeds; '-' = missing) ==")
print("symmetric RL:", {a: _n(RL_SYMM[a]) or "-" for a in ALGOS})
for e in HETERO:
    print(f"{ENV_LABEL[e]:9s} transfer:", {a: _n(RL_TRANSFER[e][a]) or "-" for a in ALGOS},
          " retune:", {a: _n(RL_RETUNE[e][a]) or "-" for a in ALGOS})
for e in ENVS:
    print(f"{ENV_LABEL[e]:9s} baselines:", {k: _n(BASE[e][k]) or "-" for k in BASELINE_KINDS})


## Helpers

In [ ]:
# --- Curve helpers -----------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from src.experiments.utils.seed_evaluation import compute_curve_stats_across_seeds
from src.analysis.utils.compute_normalized import normalize_curve, reference_value
from src.analysis.utils.compute_gap import gap_curve

def anchors_for(env):
    rnd, bso = BASE[env].get("Random"), BASE[env].get("BSOptimized")
    if not (rnd and bso):
        return None
    return reference_value(rnd), reference_value(bso)

def _seed_curves(run):
    return [s["training_metrics"] for s in run["seed_data"].values() if s["training_metrics"]]

def gap_eval_curve(run, env):
    """Optimality-gap eval curve in percent (iterations, mean%, std%), or None.
    Lead reporting metric: 0 = BS-Optimized ceiling, higher = worse."""
    if not run:
        return None
    a = anchors_for(env)
    if a is None:
        return None
    seed_runs = _seed_curves(run)
    if not seed_runs:
        return None
    it, mean, std = compute_curve_stats_across_seeds(seed_runs, key="eval_return")
    c = gap_curve(it, mean, std, a[1])           # BS-Optimized anchor only
    return c["iterations"], 100 * c["means"], 100 * c["stds"]

def norm_eval_curve(run, env):
    """Normalized-score eval curve (iterations, mean, std), or None. Registered
    metric, kept for the companion figures."""
    if not run:
        return None
    a = anchors_for(env)
    if a is None:
        return None
    seed_runs = _seed_curves(run)
    if not seed_runs:
        return None
    it, mean, std = compute_curve_stats_across_seeds(seed_runs, key="eval_return")
    c = normalize_curve(it, mean, std, a[0], a[1])
    return c["iterations"], c["means"], c["stds"]

def focus_top(curves, pad=1.3, floor_min=5.0):
    """Robust y-top for gap curves: cap to the converged regime (last 20% of
    iterations) so the early-training transient does not flatten the figure."""
    tops = []
    for it, m, s in curves:
        m, s = np.asarray(m), np.asarray(s)
        if m.size == 0:
            continue
        k = max(1, m.size // 5)
        tops.append(float(np.nanmax((m + s)[-k:])))
    return max(floor_min, pad * max(tops)) if tops else None

COLORS = {a: plt.cm.tab10.colors[i] for i, a in enumerate(["MAPPO-PS", "IPPO-PS", "CPPO",
                                                            "MAPPO-noPS", "IPPO-noPS"])}

## P1. Symmetric reference -- optimality gap (lead)

% above the BS-Optimized cost; lower is better, 0 = ceiling. Early-training transient is clipped to keep the converged regime legible.

In [ ]:
# --- P1: Symmetric reference -- optimality gap (LEAD) ------------------------
env = "symmetric"
fig, ax = plt.subplots(figsize=(7, 4.2))
curves = []
for a in ALGOS:
    cur = gap_eval_curve(RL_SYMM.get(a), env)
    if cur is None:
        continue
    it, m, s = cur
    curves.append(cur)
    col = COLORS.get(a, "C0")
    ax.plot(it, m, label=a, color=col, lw=1.7)
    ax.fill_between(it, m - s, m + s, color=col, alpha=0.18)
ax.axhline(0.0, ls="--", c="k", lw=1.0, label="BS-Optimized")
top = focus_top(curves)
if top:
    ax.set_ylim(bottom=-3, top=top)
ax.set(xlabel="Training iteration", ylabel="% above optimal cost (lower = better)",
       title="Symmetric reference")
ax.legend(fontsize=8, loc="upper right")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "p1_symmetric_gap.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "p1_symmetric_gap.png", dpi=150, bbox_inches="tight")
plt.show()

## P2. The heterogeneity gap -- optimality gap, 2x2 (lead)

Solid = re-tuned, dotted = transferred. The height of the solid curves above the 0 line is the residual optimality gap.

In [ ]:
# --- P2: Heterogeneity gap -- optimality gap, transfer (dotted) vs re-tune (solid)
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5), sharex=True, sharey=True)
all_curves = []
for ax, env in zip(axes.ravel(), HETERO):
    for a in ALGOS:
        col = COLORS.get(a, "C0")
        ct = gap_eval_curve(RL_TRANSFER[env].get(a), env)
        if ct is not None:
            all_curves.append(ct)
            ax.plot(ct[0], ct[1], ls=":", color=col, lw=1.4, alpha=0.9)
        cq = gap_eval_curve(RL_RETUNE[env].get(a), env)
        if cq is not None:
            all_curves.append(cq)
            it, m, s = cq
            ax.plot(it, m, ls="-", color=col, lw=1.7, label=a)
            ax.fill_between(it, m - s, m + s, color=col, alpha=0.15)
    ax.axhline(0.0, ls="--", c="k", lw=1.0)
    ax.set_title(ENV_LABEL[env])
    ax.grid(alpha=0.3)
top = focus_top(all_curves)
if top:
    axes[0, 0].set_ylim(bottom=-3, top=top)   # shared y
axes[0, 0].legend(fontsize=7, loc="upper right", title="re-tune (solid)\ntransfer (dotted)")
for ax in axes[-1, :]:
    ax.set_xlabel("Training iteration")
for ax in axes[:, 0]:
    ax.set_ylabel("% above optimal (lower = better)")
fig.suptitle("Heterogeneity gap: optimality gap to BS-Optimized ceiling (= 0)")
fig.tight_layout()
fig.savefig(FIG_DIR / "p2_heterogeneity_gap.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "p2_heterogeneity_gap.png", dpi=150, bbox_inches="tight")
plt.show()

## P3. Failure-mode rollouts (main text)

Per-policy behaviour of the re-tuned MAPPO-PS baseline. SKU shows per-SKU collapse; Demand shows per-warehouse scale collapse.

In [ ]:
# --- P3: Failure-mode rollouts (re-tuned MAPPO-PS) ---------------------------
# Needs ROLLOUTS paths (SETUP). Renders the full per-SKU/per-warehouse panel via
# eval_rollout_plots and displays the key panels inline.
from src.visualization.utils.eval_rollout_plots import generate_visualizations
from IPython.display import Image, display as idisplay

def render_rollout(env, show=("01_inventory.png", "05b_per_sku_fill_rate.png", "04_cost_breakdown.png")):
    p = ROLLOUTS.get(env)
    if not p:
        print(f"[skip] {env}: no rollout path set in ROLLOUTS")
        return None
    p = Path(p)
    if not (p / "episodes_data.npz").exists():
        print(f"[skip] {env}: no episodes_data.npz in {p}")
        return None
    eps = load_episodes(p)
    out = FIG_DIR / f"rollout_{ENV_TOK[env]}"
    out.mkdir(parents=True, exist_ok=True)
    generate_visualizations(eps, str(out))
    print(f"[ok] {env}: {len(eps)} episodes -> {out}")
    for png in show:
        f = out / png
        if f.exists():
            print(f"--- {ENV_LABEL[env]}: {png} ---")
            idisplay(Image(filename=str(f)))
    return out

# MAIN TEXT: SKU (per-SKU collapse) and Demand (per-warehouse scale collapse)
for env in ["sku_hetero", "demand_hetero"]:
    render_rollout(env)

## P4. Failure-mode rollouts (appendix)

In [ ]:
# --- P4 (appendix): reference + remaining failure modes ----------------------
for env in ["symmetric", "cost_hetero", "combined_hetero"]:
    render_rollout(env)

## Normalized-score companion (registered metric)

The same P1/P2 curves on the pre-registered normalized score (BS-Optimized = 1, Random = 0). Kept for the appendix and as the unit the statistical tests are run in.

In [ ]:
# --- P1 (companion): Symmetric reference -- normalized score -----------------
env = "symmetric"
fig, ax = plt.subplots(figsize=(7, 4.2))
for a in ALGOS:
    cur = norm_eval_curve(RL_SYMM.get(a), env)
    if cur is None:
        continue
    it, m, s = cur
    col = COLORS.get(a, "C0")
    ax.plot(it, m, label=a, color=col, lw=1.7)
    ax.fill_between(it, m - s, m + s, color=col, alpha=0.18)
ax.axhline(1.0, ls="--", c="k", lw=1.0, label="BS-Optimized")
ax.axhline(0.0, ls=":", c="grey", lw=1.0, label="Random")
ax.set(xlabel="Training iteration", ylabel="Normalized score", title="Symmetric reference (normalized)")
ax.set_ylim(top=1.1)
ax.legend(fontsize=8, loc="lower right")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "p1_symmetric_reference_ns.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "p1_symmetric_reference_ns.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- P2 (companion): Heterogeneity gap -- normalized score -------------------
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5), sharex=True, sharey=True)
for ax, env in zip(axes.ravel(), HETERO):
    for a in ALGOS:
        col = COLORS.get(a, "C0")
        ct = norm_eval_curve(RL_TRANSFER[env].get(a), env)
        if ct is not None:
            ax.plot(ct[0], ct[1], ls=":", color=col, lw=1.4, alpha=0.9)
        cq = norm_eval_curve(RL_RETUNE[env].get(a), env)
        if cq is not None:
            it, m, s = cq
            ax.plot(it, m, ls="-", color=col, lw=1.7, label=a)
            ax.fill_between(it, m - s, m + s, color=col, alpha=0.15)
    ax.axhline(1.0, ls="--", c="k", lw=1.0)
    ax.set_title(ENV_LABEL[env])
    ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=7, loc="lower right", title="re-tune (solid)\ntransfer (dotted)")
for ax in axes[-1, :]:
    ax.set_xlabel("Training iteration")
for ax in axes[:, 0]:
    ax.set_ylabel("Normalized score")
fig.suptitle("Heterogeneity gap: normalized to BS-Optimized = 1 (registered metric)")
fig.tight_layout()
fig.savefig(FIG_DIR / "p2_heterogeneity_gap_ns.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "p2_heterogeneity_gap_ns.png", dpi=150, bbox_inches="tight")
plt.show()